# 01 | Harmonization: one MODIS + VIIRS fire record (Steps 4-5)

Builds the harmonized monthly series for one country (2000 to the last complete month), validates it, and saves
the slide figures to `outputs/figures/`.
The code lives in `syntra/` (`pipeline.py` → `clean.py` → `harmonize.py` → `series.py`, figures in `figures.py`);
fix things there, not here.

Unit: **burned 0.1° cells** (about 11 km): grid cells with at least one vegetation-fire detection in the month,
in VIIRS S-NPP units. Run all cells. To change the country, edit the settings cell.

## 1. Settings

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets the notebook import syntra/

import pandas as pd
from syntra import figures, series
from syntra import harmonize as hz
from syntra.pipeline import load_country

COUNTRY = "Sudan"          # as FIRMS writes it, spaces become underscores
WITH_2026 = True           # needs FIRMS_MAP_KEY in .env; False = archive only (to Dec 2025)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
figures.style()

/Users/hishamalbaiti/Library/Application Support/Claude/scratch-workspaces/e459434a-1924-4487-8da4-b22c26edc019/b393a434-87af-4c7c-9250-fbebc0a02f4d/scratch-2026-11-14-b71965/syntra-firms/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Clean data (Step 3)
Fetch the FIRMS yearly files, clean them with `syntra/clean.py`, add 2026 from the API, apply the static-source mask to every year. First run: a few minutes.

In [2]:
d = load_country(COUNTRY, with_2026=WITH_2026)
veg, info = d["veg"], d["info"]
print(f"Data through {info['complete_day']:%d %b %Y}")
print(f"{info['rows_clean']:,} clean hotspots, {len(veg):,} vegetation fires")
print(f"static: {info['static_by_type']:,} by type + {info['static_by_mask']:,} by the mask ({info['mask_cells']} cells)")
veg.groupby(["family", "satellite"]).size().rename("hotspots").to_frame()

Data through 13 Nov 2026
4,462,802 clean hotspots, 4,417,341 vegetation fires
static: 33,694 by type + 11,767 by the mask (700 cells)


hotspots
family satellite          
MODIS  Aqua         536898
       Terra        385667
VIIRS  N           2157879
       N20         1336897

## 3. Harmonize (Step 4)
Burned cells per month → Terra-only months scaled to Terra + Aqua → MODIS × monthly factor = VIIRS units before 2012. After Suomi NPP's data ends, VIIRS = NOAA-20 × k.

In [3]:
cs = series.country_series(veg, info)
tab = cs["tab"]
print(f"Terra -> Terra+Aqua, annual factor {cs['k_terra_all']:.2f} (fit {hz.TERRA_CAL[0]}-{hz.TERRA_CAL[1]})")
print(f"MODIS -> VIIRS,      annual factor {cs['k_viirs_all']:.2f} (fit {hz.OVERLAP[0]}-{hz.OVERLAP[1]})")
if info.get("snpp_end") is not None:
    print(f"NOAA-20 -> S-NPP k = {info['k_n20']:.3f} (S-NPP data ends {info['snpp_end']:%d %b %Y})")
pd.DataFrame({"MODIS->VIIRS": cs["k_viirs"].round(2), "Terra->Terra+Aqua": cs["k_terra"].round(2)}).T

Terra -> Terra+Aqua, annual factor 1.53 (fit 2003-2021)
MODIS -> VIIRS,      annual factor 1.84 (fit 2012-2021)
NOAA-20 -> S-NPP k = 0.992 (S-NPP data ends 02 Nov 2026)


month,1,2,3,4,5,6,7,8,9,10,11,12
MODIS->VIIRS,1.63,1.95,2.08,2.50,3.92,5.97,6.99,6.64,3.64,1.62,1.36,1.49
Terra->Terra+Aqua,1.52,1.61,1.60,1.62,1.74,1.88,1.83,2.31,2.48,1.77,1.34,1.45


In [4]:
tab[["year", "month", "modis_cells", "viirs_cells", "harmonized_cells", "source"]].tail(14)

,year,month,modis_cells,viirs_cells,harmonized_cells,source
298,2025,9,95,272.0,272.0,VIIRS observed
299,2025,10,969,2012.0,2012.0,VIIRS observed
300,2025,11,1916,2931.0,2931.0,VIIRS observed
301,2025,12,1742,2828.0,2828.0,VIIRS observed
302,2026,1,1624,2832.0,2832.0,VIIRS observed
303,2026,2,1416,2644.0,2644.0,VIIRS observed
304,2026,3,888,1777.0,1777.0,VIIRS observed
305,2026,4,877,1909.0,1909.0,VIIRS observed
306,2026,5,387,1447.0,1447.0,VIIRS observed
307,2026,6,117,615.0,615.0,VIIRS observed


## 4. Validate (Step 5)
Leave-one-year-out on the overlap: each year's VIIRS is predicted from MODIS with factors fitted on the other years.

In [5]:
ho, val = cs["heldout"], cs["validation"]
print(f"Monthly R2: harmonized {ho['r2']:.3f} | raw MODIS {ho['r2_raw_modis']:.3f} | {ho['n_months']} months")
print(f"Median |annual error| {val['annual_error_%'].abs().median():.1f}% | "
      f"median monthly MAPE {val['monthly_MAPE_%'].median():.1f}% | median monthly r {val['monthly_r'].median():.3f}")
n20 = series.validate_n20(veg, info.get("snpp_end"))
if n20:
    print(f"NOAA-20 -> S-NPP: k {n20['k']:.3f}, {n20['n_months']} months, held-out R2 {n20['r2']:.3f}, bias {n20['bias_pct']:+.1f}%")
val

Monthly R2: harmonized 0.975 | raw MODIS 0.389 | 120 months
Median |annual error| 3.0% | median monthly MAPE 10.8% | median monthly r 0.992


NOAA-20 -> S-NPP: k 0.992, 102 months, held-out R2 0.997, bias -0.0%


,year,viirs_cells,predicted_cells,annual_error_%,monthly_MAPE_%,monthly_r
0,2012,10936,12076,10.4,20.2,0.975
1,2013,13789,14633,6.1,10.7,0.994
2,2014,11263,11018,-2.2,11.0,0.994
3,2015,12208,12327,1.0,9.8,0.995
4,2016,10966,10879,-0.8,9.8,0.993
5,2017,12230,11962,-2.2,13.3,0.988
6,2018,12653,12507,-1.2,11.8,0.991
7,2019,13781,12907,-6.3,15.4,0.992
8,2020,16304,16913,3.7,9.3,0.988
9,2021,16338,15400,-5.7,8.6,0.997


## 5. Slide figures
Saved to `outputs/figures/` (PNG, 200 dpi).

In [6]:
figures.raw_vs_harmonized(tab, COUNTRY, FIG / f"01_raw_vs_harmonized_{COUNTRY}.png");

In [7]:
figures.heldout_scatter(ho, COUNTRY, FIG / f"02_validation_scatter_{COUNTRY}.png");

In [8]:
harm = series.as_series(tab)
figures.calendar(harm, COUNTRY, FIG / f"03_calendar_{COUNTRY}.png");

In [9]:
figures.factors(cs["k_viirs"], cs["k_terra"], COUNTRY, FIG / f"04_factors_{COUNTRY}.png");

The last complete month against the same month in every year:

In [10]:
last = harm.dropna().index[-1]
same = harm[harm.index.month == last.month].dropna()
same.index = same.index.year
figures.month_bars(same, last.year, f"{COUNTRY.replace('_', ' ')}: every {last.strftime('%B')}, burned cells",
                   "burned 0.1° cells", FIG / f"05_{last.strftime('%b')}_{last.year}_{COUNTRY}.png");

This month so far (days 1 to yesterday) against the same days of every VIIRS year:

In [11]:
mtd = series.month_to_date(veg, info)
if mtd is not None:
    day = info["complete_day"]
    figures.month_bars(figures.years_series(mtd["by_year"]), day.year,
                       f"{COUNTRY.replace('_', ' ')}: {day:%B} 1–{day.day}, every VIIRS year ({day.year}: {mtd['source'].replace(' x ', ' × ')})",
                       "burned 0.1° cells", FIG / f"06_{day:%b}_1-{day.day}_{day.year}_{COUNTRY}.png");

Several countries (from `python scripts/validate_harmonization.py`): where the method works and where it reaches its limits.

In [12]:
t = ROOT / "outputs" / "tables" / "harmonization_validation.csv"
if t.exists():
    table = pd.read_csv(t)
    figures.validation_countries(table, FIG / "07_validation_countries.png")
    display(table)

,country,months,modis_cells_per_year,viirs_cells_per_year,modis_to_viirs_annual,r2_raw_modis,r2_harmonized,median_abs_annual_error_pct,median_monthly_mape_pct,median_monthly_r
0,Sudan,120,7083,13047,1.842,0.389,0.975,3.0,10.8,0.992
1,Ethiopia,120,7327,13139,1.793,0.546,0.981,4.3,10.8,0.995
2,Iraq,120,1451,3088,2.128,0.337,0.826,9.6,14.4,0.974
3,Saudi_Arabia,120,317,1661,5.245,-20.712,-0.746,7.0,19.2,0.437
